# LogiRoute AI — End-to-End Supply Chain & Logistics ML/OR Walkthrough

This notebook demonstrates the complete **LogiRoute AI** pipeline:
1. **Synthetic Multi-Echelon Supply Chain & Telemetry Generation**
2. **Probabilistic Demand Forecasting (`HistGradientBoostingRegressor` + Split Conformalized Quantile Regression — CQR)**
3. **Traffic & Weather-Aware ETA & SLA Delay Risk Prediction with Counterfactual Attribution**
4. **Capacitated Vehicle Routing with Time Windows (`CVRPTW` — Clarke-Wright Savings + 2-Opt + Inter-Route Relocate)**
5. **Multi-Echelon Stochastic Inventory Optimization (`(s, S)` Continuous Review & Newsvendor Policy)**
6. **Multivariate `IsolationForest` Anomaly Detection, MLOps Drift Governance (`PSI` / `KS-Test`), and Pareto Frontier Analysis**

In [1]:
from logiroute.pipeline.orchestrator import ControlTowerOrchestrator
from logiroute.schemas import (
    ETAPredictionRequest,
    ScenarioSimulationRequest,
    VRPOptimizationRequest,
)

orchestrator = ControlTowerOrchestrator(seed=42)
init_metrics = orchestrator.initialize()
init_metrics

## 1. Conformalized Quantile Demand Forecasting (P10 / P50 / P90)

We generate a 14-day probabilistic forecast for `SKU-PHM-101` (*Cold-Chain Biologics & Insulin Pack*) at `NODE-SLT-01` (*Salt Lake Sector V Tech & Pharma Park*), verifying calibrated 80% conformal intervals and non-crossing quantiles.

In [2]:
fc = orchestrator.run_demand_forecast(sku_id="SKU-PHM-101", hub_id="NODE-SLT-01", horizon_days=14)
print("Holdout Metrics:", fc.metrics)
print("Top Feature Importances:", fc.feature_importances)
for pt in fc.series[-5:]:
    print(f"{pt.date} | P10={pt.p10_demand:6.1f} | P50={pt.p50_demand:6.1f} | P90={pt.p90_demand:6.1f}")

## 2. Capacitated Vehicle Routing with Time Windows (`CVRPTW`) & Pareto Frontier

In [3]:
vrp_res = orchestrator.run_vrp_optimization(VRPOptimizationRequest(enable_two_opt=True))
print(vrp_res.benchmark.model_dump())

pareto_res = orchestrator.run_pareto_frontier()
for sol in pareto_res.solutions:
    tag = "[PARETO]" if sol.is_pareto_optimal else "[DOMINATED]"
    print(f"{tag:11s} {sol.policy_id}: Cost=${sol.daily_cost_usd:.0f}, CO2={sol.daily_co2_kg:.1f}kg, SLA={sol.expected_sla_otd_pct:.1f}%")

## 3. MLOps Covariate Drift Monitoring & Multivariate Anomaly Detection

In [4]:
drift_nominal = orchestrator.run_drift_evaluation("nominal")
drift_monsoon = orchestrator.run_drift_evaluation("monsoon_cov_shift")
print("Nominal Health:", drift_nominal.overall_health_score, "| Retrain:", drift_nominal.retraining_recommended)
print("Monsoon Shift Health:", drift_monsoon.overall_health_score, "| Retrain:", drift_monsoon.retraining_recommended)

anomalies = orchestrator.run_anomaly_detection(top_k=5)
for ev in anomalies.events:
    print(f"{ev.event_id} [{ev.severity}] {ev.anomaly_type} -> {ev.root_cause_signal}")